# K-fold SCVI — per-train-run geom-distance vs val LMI

For every K-fold split (built by `2026-04-27_build_kfold_splits.py`) the
compute script trains **one SCVI per training run** (13 per fold), embeds the
fold's full 30k-cell val pool with each model, and runs `latentmi` (seed 42)
against one-hot `author_day`. Each training run carries a pre-computed
geomloss energy distance to the val pool (encoded in the source filename).

This notebook reads each `summary.json` produced by
`2026-04-27_compute_kfold_scvi_lmi.py` directly, so partial results show up
as soon as individual jobs finish (no need to wait for the consolidated CSV).
It writes a single combined PNG (`kfold_scvi_lmi.png`) with one subplot per
fold of the 13-point `(edist, lmi)` scatter. SCVI train/val ELBO curves are
plotted inline only (read from each job's `training_history.csv`) and are
not saved to `final_results/`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ANALYSIS_DIR = Path('/home/igor/igor_repos/scaling_laws/Scaling-up-measurement-noise-scaling-laws/analysis')
FINAL_RESULTS_DIR = ANALYSIS_DIR / 'final_results'
FINAL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
SCVI_OUT_DIR = Path('/home/igor/igor_repos/scaling_laws/data_local/other/batch_effects/kfold/scvi')
CSV_PATH = FINAL_RESULTS_DIR / 'kfold_scvi_lmi.csv'

FIG_DPI = 300

summary_paths = sorted(SCVI_OUT_DIR.glob('*-fold/*/summary.json'))
print(f'Found {len(summary_paths)} summary.json files under {SCVI_OUT_DIR}')

rows = []
for p in summary_paths:
    d = json.loads(p.read_text())
    if d.get('lmi') is None:
        continue
    rows.append({
        'fold': d['fold'],
        'run_id': d['run_id'],
        'val_ids': ','.join(d.get('val_ids', [])),
        'edist_train_val': d['edist_train_val'],
        'lmi': d['lmi'],
        'lmi_seed': d['lmi_seed'],
        'n_train_cells': d.get('n_train_cells'),
        'n_val_cells': d.get('n_val_cells'),
        'scvi_train_seconds': d.get('scvi_train_seconds'),
    })

df = pd.DataFrame(rows)
if df.empty:
    raise SystemExit(
        f'No completed jobs found under {SCVI_OUT_DIR}. '
        f'Run analysis/2026-04-27_compute_kfold_scvi_lmi.py first.'
    )
df = df.sort_values(['fold', 'edist_train_val']).reset_index(drop=True)
df.to_csv(CSV_PATH, index=False)
print(f'Wrote {len(df)} rows to {CSV_PATH}')
df

In [ ]:
folds_sorted = sorted(df['fold'].unique())
n_folds = len(folds_sorted)
fig, axes = plt.subplots(1, n_folds, figsize=(6.5 * n_folds, 4.5),
                         dpi=FIG_DPI, squeeze=False)
axes = axes[0]

for ax, fold in zip(axes, folds_sorted):
    sub = df[df['fold'] == fold].sort_values('edist_train_val').reset_index(drop=True)
    val_ids = sub['val_ids'].iloc[0]

    x = sub['edist_train_val'].to_numpy(dtype=float)
    y = sub['lmi'].to_numpy(dtype=float)

    ax.scatter(x, y, s=80, c='C0', edgecolors='k', linewidths=0.6, zorder=3)
    for run_id, xi, yi in zip(sub['run_id'], x, y):
        ax.annotate(run_id, xy=(xi, yi), xytext=(5, 4),
                    textcoords='offset points', fontsize=8)

    if len(x) >= 2 and np.ptp(x) > 0:
        slope, intercept = np.polyfit(x, y, deg=1)
        xs = np.linspace(x.min(), x.max(), 100)
        ax.plot(xs, slope * xs + intercept, color='C3', linestyle='--',
                linewidth=1.0, alpha=0.8, zorder=2,
                label=f'fit: y = {slope:.3f} x + {intercept:.3f}')
        r = float(np.corrcoef(x, y)[0, 1])
        ax.text(0.02, 0.97, f'Pearson r = {r:.3f}\nn = {len(x)}',
                transform=ax.transAxes, va='top', ha='left',
                fontsize=9,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8,
                          edgecolor='0.8'))
        ax.legend(loc='lower left', fontsize=8, frameon=True)

    ax.set_xlabel('train↔val energy distance (PCA-50, geomloss)')
    ax.set_ylabel('LMI(SCVI val embedding ; author_day)')
    ax.set_title(f'{fold} — val_ids = {val_ids}', fontsize=11)
    ax.grid(alpha=0.3)

fig.tight_layout()
out_png = FINAL_RESULTS_DIR / 'kfold_scvi_lmi.png'
fig.savefig(out_png, dpi=FIG_DPI, bbox_inches='tight')
print(f'Saved {out_png}')
plt.show()
plt.close(fig)

## SCVI training curves

One small-multiples figure per fold: each subplot is a single training run
from that fold, showing `elbo_train` and `elbo_validation` versus epoch as
logged by scvi-tools' Lightning trainer (read from each job's
`training_history.csv`). Subplots are ordered by `edist_train_val` so the
leftmost / topmost runs are the closest to the val pool. Displayed inline
only — not saved to `final_results/`.

In [ ]:
TRAIN_METRIC = 'elbo_train'
VAL_METRIC = 'elbo_validation'

for fold, sub in df.groupby('fold', sort=True):
    sub = sub.sort_values('edist_train_val').reset_index(drop=True)
    n = len(sub)
    n_cols = 4
    n_rows = int(np.ceil(n / n_cols))

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(3.2 * n_cols, 2.4 * n_rows),
        dpi=FIG_DPI, sharex=False, sharey=False,
    )
    axes = np.atleast_2d(axes).reshape(n_rows, n_cols)

    for ax in axes.ravel():
        ax.set_visible(False)

    missing = []
    for i, row in sub.iterrows():
        ax = axes.ravel()[i]
        ax.set_visible(True)
        run_id = row['run_id']
        edist = row['edist_train_val']
        hist_path = SCVI_OUT_DIR / fold / run_id / 'training_history.csv'
        if not hist_path.exists():
            ax.text(0.5, 0.5, 'no training_history.csv\n(re-run compute script)',
                    ha='center', va='center', fontsize=8, transform=ax.transAxes)
            ax.set_title(f'{run_id}  edist={edist:.3f}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
            missing.append(run_id)
            continue
        h = pd.read_csv(hist_path, index_col=0)
        x = h.index.to_numpy()
        if TRAIN_METRIC in h.columns:
            y_tr = h[TRAIN_METRIC].to_numpy(dtype=float)
            mask = np.isfinite(y_tr)
            ax.plot(x[mask], y_tr[mask], color='C0', linewidth=1.2,
                    label='train')
        if VAL_METRIC in h.columns:
            y_va = h[VAL_METRIC].to_numpy(dtype=float)
            mask = np.isfinite(y_va)
            ax.plot(x[mask], y_va[mask], color='C3', linewidth=1.2,
                    linestyle='--', label='val')
        ax.set_title(f'{run_id}  edist={edist:.3f}', fontsize=9)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)
        if i == 0:
            ax.legend(fontsize=7, loc='upper right')

    fig.suptitle(
        f'SCVI ELBO curves \u2014 {fold} \u2014 val_ids = {sub["val_ids"].iloc[0]}',
        fontsize=11,
    )
    fig.supxlabel('epoch', fontsize=10)
    fig.supylabel('ELBO (lower = better)', fontsize=10)
    fig.tight_layout(rect=[0.02, 0.02, 1, 0.96])
    if missing:
        print(f'  ({len(missing)} runs missing training_history.csv: {missing})')
    plt.show()
    plt.close(fig)

In [ ]:
df[['fold', 'run_id', 'val_ids', 'edist_train_val', 'lmi', 'n_val_cells']]